# Medical RAG assistant: 2026 rebuild with a valid evaluation

**Author:** Mersedyes Livingston · rebuilt November 2026 · original project completed June 2025

> **Status: not yet executed.** This notebook needs a GPU runtime (Colab T4 or better) and your own copy of the reference manual, which is copyrighted and not redistributed. Outputs are committed after it is run.
>
> Research prototype; not medical advice.

The original notebook was not recovered. This rebuild uses the same system as the 2025 presentation:
- Mistral-7B-Instruct as the generator.
- 500-character chunks with 100 overlap.
- all-MiniLM-L6-v2 embeddings, stored in ChromaDB.
- Top-5 retrieval.

It fixes the evaluation problems found in the [2026 re-audit](../README.md):

| 2025 | 2026 rebuild |
|---|---|
| The baseline's groundedness was scored against context it never saw | All three systems (base, prompt-engineered, RAG) are judged against **the same retrieved passages** for each question |
| Judge model unrecorded, possibly the generator itself | The judge is a **different model family** (Llama 3.1 8B), named in the config, and sees no system labels |
| No human check on the judge | A blind CSV for human rating, with weighted Cohen's κ against the judge once ratings exist |
| Retrieval never evaluated | Hit@k against reference pages you label |
| n = 5, "significant" | Paired per-question differences with bootstrap intervals. The 5 questions are a starting set; extend to 50+ |


## 0. Install (Colab, GPU runtime). The CUDA build of `llama-cpp-python` takes about 10 minutes

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python
!pip install -q sentence-transformers chromadb langchain-text-splitters pypdf huggingface_hub scikit-learn pandas

## 1. Configuration (everything that affects results is recorded)

In [ ]:
import gc, json, platform, re
from pathlib import Path

import numpy as np
import pandas as pd

MANUAL_PDF = Path("medical_diagnosis_manual.pdf")   # upload your copy to the Colab session; not redistributed
GEN = ("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf")
JUDGE = ("bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf")
assert GEN[0].split("/")[1].split("-")[0] != JUDGE[0].split("/")[1].split("-")[0], "judge must not be the generator's model family"
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 100, 5
TEMPERATURE, TOP_P, MAX_TOKENS, N_CTX, SEED = 0.0, 0.9, 512, 4096, 1
OUT_DIR = Path("rag_outputs"); OUT_DIR.mkdir(exist_ok=True)

SYSTEM_PROMPT = (  # the 2025 system prompt
    "You are an expert medical assistant that provides information based on the provided context. "
    "You are going to be asked medical questions. You need to provide the correct and accurate responses "
    "to the questions. Do not make up information you don't know. If there is something you do not know, "
    "please state that you don't know the answer to that question.")

## 2. Load the manual and chunk it (page numbers are kept for citation and retrieval scoring)

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

if not MANUAL_PDF.is_file():
    raise FileNotFoundError(f"Upload the manual as {MANUAL_PDF} (it is not included in the repository)")
reader = PdfReader(str(MANUAL_PDF))
pages = [(i + 1, (p.extract_text() or "")) for i, p in enumerate(reader.pages)]
pages = [(n, t) for n, t in pages if t.strip()]
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = [{"id": f"p{n}-c{j}", "page": n, "text": c} for n, t in pages for j, c in enumerate(splitter.split_text(t))]
print(f"{len(reader.pages):,} pages ({len(pages):,} with text) -> {len(chunks):,} chunks")

## 3. Embed and index (ChromaDB, cosine distance)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=str(OUT_DIR / "chroma"))
col = client.get_or_create_collection("manual", metadata={"hnsw:space": "cosine"})
if col.count() != len(chunks):  # (re)build when the chunking changed
    client.delete_collection("manual")
    col = client.create_collection("manual", metadata={"hnsw:space": "cosine"})
    for i in range(0, len(chunks), 512):
        batch = chunks[i:i + 512]
        col.add(ids=[c["id"] for c in batch], documents=[c["text"] for c in batch],
                metadatas=[{"page": c["page"]} for c in batch],
                embeddings=emb.encode([c["text"] for c in batch], normalize_embeddings=True).tolist())

def retrieve(question, k=TOP_K):
    r = col.query(query_embeddings=emb.encode([question], normalize_embeddings=True).tolist(), n_results=k)
    return [{"text": d, "page": m["page"], "distance": float(s)}
            for d, m, s in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

def context_text(ctx):
    return "\n\n".join(f"[p.{c['page']}] {c['text']}" for c in ctx)

print("indexed chunks:", col.count())

## 4. Questions
These are the 5 questions from 2025. Fill `reference_pages` with the manual pages that answer each one, so retrieval can be scored. Then extend the set (target: 50+, stratified by question type).

In [ ]:
QUESTIONS = [
    {"id": "Q1", "topic": "sepsis", "reference_pages": [],
     "question": "What is the protocol for managing sepsis in a critical care unit?"},
    {"id": "Q2", "topic": "appendicitis", "reference_pages": [],
     "question": "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"},
    {"id": "Q3", "topic": "hair loss", "reference_pages": [],
     "question": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"},
    {"id": "Q4", "topic": "brain injury", "reference_pages": [],
     "question": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"},
    {"id": "Q5", "topic": "leg fracture", "reference_pages": [],
     "question": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"},
]
CONTEXTS = {q["id"]: retrieve(q["question"]) for q in QUESTIONS}
for q in QUESTIONS:
    print(q["id"], "retrieved pages:", [c["page"] for c in CONTEXTS[q["id"]]])

## 5. Generate answers: base model, prompt-engineered, and RAG (temperature 0 for comparability)

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

def load_model(repo_file):
    path = hf_hub_download(repo_id=repo_file[0], filename=repo_file[1])
    return Llama(model_path=path, n_ctx=N_CTX, n_gpu_layers=-1, seed=SEED, verbose=False)

def chat(model, user, system=None, merge_system=True, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    # Mistral's chat template has no system role, so the system prompt is merged into the user turn.
    if system and merge_system:
        user, system = f"{system}\n\n{user}", None
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    out = model.create_chat_completion(messages=messages, temperature=temperature, top_p=TOP_P,
                                       max_tokens=max_tokens, seed=SEED)
    return out["choices"][0]["message"]["content"].strip()

llm = load_model(GEN)
answers = []
for q in QUESTIONS:
    rag_prompt = (f"Context:\n{context_text(CONTEXTS[q['id']])}\n\nQuestion: {q['question']}\n"
                  "Answer using only the context above and cite page numbers like [p.123].")
    answers += [{"id": q["id"], "system": "base", "answer": chat(llm, q["question"])},
                {"id": q["id"], "system": "prompted", "answer": chat(llm, q["question"], SYSTEM_PROMPT)},
                {"id": q["id"], "system": "rag", "answer": chat(llm, rag_prompt, SYSTEM_PROMPT)}]
answers = pd.DataFrame(answers)
answers.to_csv(OUT_DIR / "answers.csv", index=False)
del llm; gc.collect()
answers.groupby("system").answer.apply(lambda s: s.str.len().mean()).rename("mean answer length (chars)")

## 6. Retrieval quality: hit@k (computed only for questions with labeled reference pages)

In [ ]:
retrieval = pd.DataFrame([{"id": q["id"], "labeled": bool(q["reference_pages"]),
                           f"hit@{TOP_K}": (bool(set(q["reference_pages"]) & {c["page"] for c in CONTEXTS[q["id"]]})
                                            if q["reference_pages"] else None)} for q in QUESTIONS])
retrieval

## 7. LLM judge: blind and grounded in the same passages
- The judge model is from a different family than the generator.
- It never sees which system wrote the answer.
- For every system, groundedness is scored against the **same retrieved passages**. That is what makes the base-vs-RAG comparison fair.

In [ ]:
RUBRIC = {
    "groundedness": ("Rate 0-3 how well EVERY claim in the ANSWER is supported by the REFERENCE PASSAGES. "
                     "3 = all claims supported with specific evidence; 2 = mostly supported; 1 = partly supported; "
                     "0 = not supported. Judge only against the passages, not your own knowledge."),
    "relevance": ("Rate 0-3 how directly and completely the ANSWER responds to the QUESTION. "
                  "3 = fully relevant and complete; 2 = mostly; 1 = partially; 0 = irrelevant or off-topic."),
}
judge = load_model(JUDGE)

def judge_score(metric, question, answer, ctx):
    prompt = (f"{RUBRIC[metric]}\n\nQUESTION:\n{question}\n\nREFERENCE PASSAGES:\n{context_text(ctx)}\n\n"
              f"ANSWER:\n{answer}\n\nRespond only with JSON: {{\"score\": <0-3>, \"reason\": \"<one sentence>\"}}")
    raw = chat(judge, prompt, "You are a strict, impartial evaluator.", merge_system=False, max_tokens=200)
    m = re.search(r'"score"\s*:\s*([0-3])', raw)
    return (int(m.group(1)) if m else None), raw

qtext = {q["id"]: q["question"] for q in QUESTIONS}
rows = []
for r in answers.itertuples():
    for metric in RUBRIC:
        score, raw = judge_score(metric, qtext[r.id], r.answer, CONTEXTS[r.id])
        rows.append({"id": r.id, "system": r.system, "metric": metric, "score": score, "judge_raw": raw})
scores = pd.DataFrame(rows)
del judge; gc.collect()
print("unparseable judge outputs:", int(scores.score.isna().sum()))
scores.pivot_table(index="system", columns="metric", values="score", aggfunc="mean").loc[["base", "prompted", "rag"]]

## 8. Paired differences (RAG − base) with bootstrap intervals over questions

In [ ]:
def paired(metric, a="rag", b="base", n=5000, seed=SEED):
    w = scores[scores.metric == metric].pivot(index="id", columns="system", values="score").dropna()
    d = (w[a] - w[b]).to_numpy()
    rng = np.random.default_rng(seed)
    boots = [rng.choice(d, len(d)).mean() for _ in range(n)]
    return {"metric": metric, "n_questions": len(d), "mean_diff": d.mean(), "ci95": np.percentile(boots, [2.5, 97.5]).round(2).tolist(),
            "rag_better": int((d > 0).sum()), "tied": int((d == 0).sum()), "rag_worse": int((d < 0).sum())}

pd.DataFrame([paired(m) for m in RUBRIC])

## 9. Human calibration of the judge
Rate `human_rating_blind.csv` without looking at `human_rating_key.csv`. Ideally a clinician does this. Save the file as `human_ratings.csv` and rerun this cell to get the judge–human agreement.

In [ ]:
from sklearn.metrics import cohen_kappa_score

items = answers.merge(pd.Series(qtext, name="question"), left_on="id", right_index=True)
items = items.sample(frac=1, random_state=SEED).reset_index(drop=True)
items["item"] = [f"item-{i:03d}" for i in range(len(items))]
items[["item", "question", "answer"]].assign(groundedness_human="", relevance_human="").to_csv(OUT_DIR / "human_rating_blind.csv", index=False)
items[["item", "id", "system"]].to_csv(OUT_DIR / "human_rating_key.csv", index=False)

rated = OUT_DIR / "human_ratings.csv"
if rated.is_file():
    h = pd.read_csv(rated).merge(items[["item", "id", "system"]], on="item")
    for metric in RUBRIC:
        j = scores[scores.metric == metric][["id", "system", "score"]]
        m = h.merge(j, on=["id", "system"]).dropna(subset=[f"{metric}_human", "score"])
        print(f"{metric}: weighted kappa (judge vs human) = "
              f"{cohen_kappa_score(m[f'{metric}_human'].astype(int), m['score'].astype(int), weights='quadratic'):.2f} (n={len(m)})")
else:
    print(f"wrote {OUT_DIR/'human_rating_blind.csv'}; add ratings and save as {rated.name} to compute agreement")

## 10. Save results with full configuration

In [ ]:
import chromadb, llama_cpp, sentence_transformers
results = {
    "config": {"generator": GEN, "judge": JUDGE, "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
               "top_k": TOP_K, "temperature": TEMPERATURE, "top_p": TOP_P, "seed": SEED, "n_chunks": len(chunks)},
    "judge_means": scores.pivot_table(index="system", columns="metric", values="score", aggfunc="mean").to_dict(),
    "paired_rag_minus_base": [paired(m) for m in RUBRIC],
    "retrieval": retrieval.to_dict(orient="records"),
    "versions": {"python": platform.python_version(), "llama_cpp": llama_cpp.__version__, "chromadb": chromadb.__version__,
                 "sentence_transformers": sentence_transformers.__version__, "pandas": pd.__version__},
}
(OUT_DIR / "results.json").write_text(json.dumps(results, indent=2, default=str) + "\n")
scores.to_csv(OUT_DIR / "judge_scores.csv", index=False)
print("wrote", sorted(p.name for p in OUT_DIR.iterdir()))